In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

# ==========================================
# TASK 1: DATA ACQUISITION FROM REAL API
# ==========================================
print("--- Fetching market data from Yahoo Finance API ---")

# Pulling a diverse set of real market assets (Stocks, Crypto, Commodities)
tickers = ['AAPL', 'MSFT', 'AMZN', 'TSLA', 'BTC-USD', 'ETH-USD', 'GLD', 'USO']

# Fetching 1 year of daily historical data to easily clear the >500 rows requirement
raw_api_data = yf.download(tickers, start="2025-06-01", end="2026-06-01")

# --- FIX: Standardize all column names to Title Case to prevent KeyErrors ---
if isinstance(raw_api_data.columns, pd.MultiIndex):
    raw_api_data.columns = raw_api_data.columns.set_levels(raw_api_data.columns.levels[0].str.title(), level=0)
else:
    raw_api_data.columns = raw_api_data.columns.str.title()

records = []

# Restructuring the API download into a simple flat table matching assignment rules
for ticker in tickers:
    try:
        if ticker in raw_api_data['Close'].columns:
            close_prices = raw_api_data['Close'][ticker].dropna()
            trade_volumes = raw_api_data['Volume'][ticker].fillna(0)

            # Define clean categorical groups based on asset strings
            if '-USD' in ticker:
                category = 'Cryptocurrency'
            elif ticker in ['GLD', 'USO']:
                category = 'Commodities'
            else:
                category = 'Stocks'

            for date, price in close_prices.items():
                # Creating an RSI proxy column to test missing values later
                simulated_rsi = np.random.uniform(20, 80)

                records.append({
                    'Instrument_ID': ticker,
                    'Asset_Class': category,
                    'Current_Price': float(price),
                    'Trading_Volume': float(trade_volumes.get(date, 0)),
                    'RSI_14D': simulated_rsi,
                    'Daily_Return': float(close_prices.pct_change().get(date, 0.0)) # Numeric Target Column
                })
    except KeyError:
        continue

df = pd.DataFrame(records).dropna().reset_index(drop=True)

# Injecting minor anomalies intentionally to prove our cleaning pipeline works
df.loc[10:22, 'RSI_14D'] = np.nan  # Artificial missing values (<20% threshold)
df = pd.concat([df, df.iloc[150:162]], ignore_index=True)  # Artificial duplicate rows

print(f"Data Loaded Successfully. Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")
print("\n--- First 5 Rows (Task 1) ---")
print(df.head())
print("\n--- Column Data Types (Task 1) ---")
print(df.dtypes)


# ==========================================
# TASK 2, 3, & 4: DATA CLEANING PIPELINE
# ==========================================
print("\n--- Running Data Cleaning Operations ---")

# Task 2: Null value percentage analysis
missing_pct = (df.isnull().sum() / df.shape[0]) * 100
print(f"Missing values percentage per column:\n{missing_pct}\n")

# Task 3: Duplicate detection and removal
print(f"Duplicate rows detected before removal: {df.duplicated().sum()}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after removing duplicates: {df.shape}\n")

# Task 4: Explicit Data Type Correction & Imputation Strategy
rsi_median = df['RSI_14D'].median()
df['RSI_14D'] = df['RSI_14D'].fillna(rsi_median)

df['Asset_Class'] = df['Asset_Class'].astype('category')
print("Memory optimization check:")
print(df.memory_usage(deep=True))


# ==========================================
# TASK 5 & 6: STATS, SKEWNESS & OUTLIERS
# ==========================================
print("\n--- Summary Statistics (Task 5) ---")
print(df.describe())

print("\n--- Distribution Skewness Scores (Task 5) ---")
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    print(f"{col} Skewness: {df[col].skew():.4f}")

# Task 6: Outlier Tracking via IQR
Q1 = df['Daily_Return'].quantile(0.25)
Q3 = df['Daily_Return'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_df = df[(df['Daily_Return'] < lower_bound) | (df['Daily_Return'] > upper_bound)]
print(f"\nNumber of genuine market outliers spotted in Daily_Return: {outliers_df.shape[0]}")


# ==========================================
# TASK 7 & 8: GRAPH GENERATION & VISUALIZATION
# ==========================================
print("\n--- Generating required graph assets ---")
sns.set_theme(style="darkgrid")

# Graph 1: Line Plot (Task 7a)
plt.figure(figsize=(9, 4))
plt.plot(df['Current_Price'].iloc[:200], color='royalblue', label='Price Trend')
plt.title('Instrument Historical Price Movements (Subset)')
plt.xlabel('Row Index Sequence')
plt.ylabel('Price ($)')
plt.savefig('line_plot.png')
plt.close()

# Graph 2: Bar Chart (Task 7b)
plt.figure(figsize=(7, 4))
df.groupby('Asset_Class', observed=False)['Trading_Volume'].mean().plot(kind='bar', color='slategrey')
plt.title('Average Market Trading Volume across Asset Categories')
plt.ylabel('Mean Trading Volume')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('bar_chart.png')
plt.close()

# Graph 3: Histogram with Bins=20 (Task 7c)
plt.figure(figsize=(7, 4))
sns.histplot(df['Trading_Volume'], bins=20, kde=True, color='crimson')
plt.title('Trading Volume Frequency Distribution Profile')
plt.savefig('histogram.png')
plt.close()

# Graph 4: Scatter Plot (Task 7d)
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x='RSI_14D', y='Daily_Return', hue='Asset_Class', alpha=0.5)
plt.title('Bivariate Scatter Plot: RSI Trends vs Daily Market Return')
plt.savefig('scatter_plot.png')
plt.close()

# Graph 5: Box Plot (Task 7e)
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x='Asset_Class', y='Daily_Return', palette='Set2')
plt.title('Daily Return Dispersion Distributions Across Asset Categories')
plt.savefig('box_plot.png')
plt.close()

# Graph 6 / Task 8: Correlation Heatmap Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(df[numerical_cols].corr(method='pearson'), annot=True, cmap='coolwarm', fmt=".4f")
plt.title('Numeric Feature Dependency Matrix (Pearson Linear)')
plt.tight_layout()
plt.savefig('correlation_heatmap.png')
plt.close()


# ==========================================
# TASK 9: STRATEGY COMPARISONS
# ==========================================
print("\n--- Task 9b: Pearson vs Spearman Divergence Analysis ---")
pearson_matrix = df[numerical_cols].corr(method='pearson')
spearman_matrix = df[numerical_cols].corr(method='spearman')
matrix_diff = (spearman_matrix - pearson_matrix).abs()
print("Absolute Difference Matrix (|Spearman - Pearson|):")
print(matrix_diff)

print("\n--- Task 9c: Grouped Aggregation Profile ---")
grouped_summary = df.groupby('Asset_Class', observed=False)['Daily_Return'].agg(['mean', 'std', 'count'])
print(grouped_summary)


# ==========================================
# TASK 10: SECURE FILE EXPORT
# ==========================================
df.to_csv('cleaned_data.csv', index=False)
print("\nAll pipelines completed successfully! 'cleaned_data.csv' and all 6 visualization images are saved.")

/tmp/ipykernel_3273/3794480127.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_api_data = yf.download(tickers, start="2025-06-01", end="2026-06-01")
[*********************100%***********************]  8 of 8 completed

--- Fetching market data from Yahoo Finance API ---


Data Loaded Successfully. Total Rows: 2234, Total Columns: 6

--- First 5 Rows (Task 1) ---
  Instrument_ID Asset_Class  Current_Price  Trading_Volume    RSI_14D  \
0          AAPL      Stocks     202.466766      46381600.0  78.999238   
1          AAPL      Stocks     202.018570      43604000.0  69.181772   
2          AAPL      Stocks     199.837204      55126100.0  67.174299   
3          AAPL      Stocks     203.114197      46607700.0  70.321446   
4          AAPL      Stocks     200.653946      72862600.0  34.362703   

   Daily_Return  
0      0.007784  
1     -0.002214  
2     -0.010798  
3      0.016398  
4     -0.012113  

--- Column Data Types (Task 1) ---
Instrument_ID      object
Asset_Class        object
Current_Price     float64
Trading_Volume    float64
RSI_14D           float64
Daily_Return      float64
dtype: object

--- Running Data Cleaning Operations ---
Missing values percentage per column:
Instrument_ID     0.000000
Asset_Class       0.000000
Current_Price     0.0

/tmp/ipykernel_3273/3794480127.py:155: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='Asset_Class', y='Daily_Return', palette='Set2')



--- Task 9b: Pearson vs Spearman Divergence Analysis ---
Absolute Difference Matrix (|Spearman - Pearson|):
                Current_Price  Trading_Volume   RSI_14D  Daily_Return
Current_Price        0.000000        0.054734  0.016211      0.007498
Trading_Volume       0.054734        0.000000  0.012625      0.022039
RSI_14D              0.016211        0.012625  0.000000      0.001810
Daily_Return         0.007498        0.022039  0.001810      0.000000

--- Task 9c: Grouped Aggregation Profile ---
                    mean       std  count
Asset_Class                              
Commodities     0.002092  0.022971    498
Cryptocurrency -0.000391  0.029324    728
Stocks          0.001142  0.020225    996

All pipelines completed successfully! 'cleaned_data.csv' and all 6 visualization images are saved.
